# Cooldown Test of the Pilot Runs

This notebook measures how many training steps the pilot runs needed. All pilot runs used a cosine learning rate schedule over 45,000 steps. Under a cosine schedule the late part of the curve flattens partly because the learning rate decays, so a flat curve does not show that a shorter run would have reached the same error.

The test therefore starts short cooldowns from the milestone checkpoints of the pilot, saved at 11,250, 22,500 and 33,750 steps. From each milestone, training continues for 15% of the milestone step while the learning rate falls from its value on the cosine schedule to the minimum learning rate, with the (1 - sqrt) shape. The end of each branch estimates the error of a run whose budget ends there. The branches are compared with the full 45,000-step run by the paired difference of the normalised RMSE over the 32 validation windows, with a 95% bootstrap interval. The smallest budget whose interval stays below a fixed tolerance is the budget that the run needed.

The test covers `pilot_spheridiff_b4` and the three WIND sizes. Every branch keeps the seed of the pilot (42) and continues the same batch stream from the milestone step, so the branches differ from the full run only in the learning rate schedule.

The notebook is written for Google Colab with one T4 GPU. The milestone checkpoints and the ERA5 cache are read from one archive on Google Drive, and the repository code is cloned from GitHub. The results are packed into one archive on Google Drive, which is refreshed during the run and lets a new session skip the branches that already finished.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import torch
from tqdm import tqdm

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive"
UPLOAD_ARCHIVE = DRIVE_ROOT / "spheridiff_colab_cooldown.zip"
RESULTS_ARCHIVE = DRIVE_ROOT / "spheridiff_cooldown_results.zip"
WORK_ROOT = Path("/content")
INPUT_ROOT = WORK_ROOT / "inputs"
RUNS_ROOT = WORK_ROOT / "runs"
EXPERIMENTS = ["pilot_spheridiff_b4", "pilot_wind_s", "pilot_wind_m", "pilot_wind_l"]
COOLDOWN_STARTS = [11250, 22500, 33750]
COOLDOWN_FRACTION = 0.15
TOLERANCE = 0.002

## Inputs and hardware

The archive `spheridiff_colab_cooldown.zip` in the root of Google Drive holds the ERA5 cache at 5.6 degrees (`spheridiff/era5_5p6deg`) and, for each pilot run, the three milestone checkpoints together with the configuration snapshot and the evaluation logs of the full run (`spheridiff/pilot/<run>`). The archive is copied to the local disk of the Colab machine and unpacked there, because reading many large files from mounted Drive is slow.

When the results archive of an earlier session exists on Drive, it is unpacked into the run folder first. A branch whose summary shows that it reached its last step is then skipped, so an interrupted test continues with the remaining branches.

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`). The cooldown needs a CUDA device, and the notebook stops when none is available.

In [ ]:
from google.colab import drive

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"


def unpack(archive, target):
    with zipfile.ZipFile(archive) as handle:
        for member in tqdm(handle.infolist(), desc=f"unpacking {archive.name}"):
            handle.extract(member, target)


drive.mount(str(DRIVE_MOUNT))
if not (INPUT_ROOT / "spheridiff").exists():
    local_copy = WORK_ROOT / UPLOAD_ARCHIVE.name
    shutil.copy(UPLOAD_ARCHIVE, local_copy)
    unpack(local_copy, INPUT_ROOT)
    local_copy.unlink()
if RESULTS_ARCHIVE.exists() and not RUNS_ROOT.exists():
    unpack(RESULTS_ARCHIVE, RUNS_ROOT)
if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
sys.path.insert(0, str(CODE_ROOT / "src"))
CACHE_ROOT = INPUT_ROOT / "spheridiff" / "era5_5p6deg"
REFERENCE_ROOT = INPUT_ROOT / "spheridiff" / "pilot"
if not torch.cuda.is_available():
    raise RuntimeError("the cooldown test needs a GPU runtime")
print("code:", CODE_ROOT)
print("data:", CACHE_ROOT, sorted(os.listdir(CACHE_ROOT / "frames")))
print("milestones:", REFERENCE_ROOT, sorted(os.listdir(REFERENCE_ROOT)))
print("gpu:", torch.cuda.get_device_name(0))

## Running the branches

Each pilot run is one job, a separate process that loads the data once and then runs its three branches one after another, starting from the earliest milestone. The jobs run one after another on the single GPU, because processes that share one GPU are slowed by time slicing and finish no earlier in total. The notebook prints every line that a job writes as soon as it appears, and the same lines are kept in `runs/_logs/parallel.log`.

A branch is written to its own run folder, named after the pilot run and the milestone step, for example `pilot_spheridiff_b4_cooldown_022500`. Its only evaluation is at the last step, with the same windows, ensemble members and sampling steps as the evaluation during the pilot. When a branch has finished, its large training checkpoints are deleted and only the EMA weights and the logs are kept.

At a measured speed of about 0.51 s per step for `spheridiff_b4` and 0.11 to 0.14 s per step for the WIND sizes on a T4, the twelve branches take about two and a half hours in total. The branches and the packing of the results share one cell with a `try` and a `finally`, so the summary, the figure and the archive are produced even when a job fails or the cell is interrupted. The archive is refreshed every 15 minutes during the run.

In [ ]:
from spheridiff.analysis.cooldown import plot_cooldowns, write_cooldown_summary
from spheridiff.kaggle.finalise import finalise, guarded, refresh_archive
from spheridiff.kaggle.jobs import cooldown_job, place_queues
from spheridiff.kaggle.runner import ParallelRunner, RunContext
from spheridiff.training.cooldown import branch_name

LOGS = RUNS_ROOT / "_logs"
CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, LOGS, CACHE_ROOT)
JOBS = [cooldown_job(name, REFERENCE_ROOT, COOLDOWN_STARTS, COOLDOWN_FRACTION) for name in EXPERIMENTS]
BRANCHES = [branch_name(name, step) for name in EXPERIMENTS for step in COOLDOWN_STARTS]
FIGURE = RUNS_ROOT / "cooldown_curves.png"


def refresh():
    return refresh_archive(RUNS_ROOT, RESULTS_ARCHIVE)


def summarise():
    print(write_cooldown_summary(RUNS_ROOT, REFERENCE_ROOT, EXPERIMENTS, COOLDOWN_STARTS, TOLERANCE))
    plot_cooldowns(RUNS_ROOT, REFERENCE_ROOT, EXPERIMENTS, COOLDOWN_STARTS, FIGURE)


RUNNER = None
try:
    RUNNER = ParallelRunner(place_queues([(0, JOBS)], 1), CONTEXT, refresh=refresh, refresh_every=900)
    RUNNER.run()
finally:
    OUTCOMES = RUNNER.outcomes() if RUNNER else []
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    guarded(RUNS_ROOT, "cooldown_summary", summarise)
    finalise(RUNS_ROOT, RESULTS_ARCHIVE, OUTCOMES, BRANCHES)
print(OUTCOMES)
print(RESULTS_ARCHIVE, f"{RESULTS_ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The summary lists, for each pilot run, the error of the full 45,000-step run and, for each branch, the error at the milestone, the error after the cooldown, and the paired difference to the full run with its 95% interval. A branch matches the full run when the upper end of the interval is at most the tolerance of 0.002 in normalised RMSE. The last line of each run gives the smallest cooled budget that matches the full run. When no branch matches, the run still improved after the last milestone, and its needed budget lies above about 39,000 steps.

The figure shows the validation error of each full cosine run against the training steps, with one line for each branch from its milestone to its cooled end. The status file lists the duration and the return code of each job.

In [ ]:
import json

summary = RUNS_ROOT / "cooldown_summary.txt"
print(summary.read_text(encoding="utf-8") if summary.exists() else "no summary, see runs/_logs")
for name in BRANCHES:
    path = RUNS_ROOT / name / "summary.json"
    print(name, json.loads(path.read_text()) if path.exists() else "not finished")

In [ ]:
from IPython.display import Image, display

if FIGURE.exists():
    display(Image(filename=str(FIGURE)))